# Terminal & Shell Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: Know your shell

Check which shell you're running:

In [ ]:
```bash

echo $SHELL

In [ ]:
```

Most systems use `bash` or `zsh`. Both work fine. The commands in this course work in either.

Key things to know:

In [ ]:
```bash

# Move around

cd ~/projects/ai-engineering-from-scratch

pwd

ls -la

# History search (most useful shortcut you'll learn)

# Ctrl+R then type part of a previous command

# Press Ctrl+R again to cycle through matches

# Clear terminal

clear   # or Ctrl+L

# Cancel a running command

# Ctrl+C

# Suspend a running command (resume with fg)

# Ctrl+Z

In [ ]:
```

### Step 2: Piping and redirects

Piping connects commands together. This is how you process logs, filter output, and chain tools. You will use this constantly.

In [ ]:
```bash

# Count how many times "loss" appears in a log

cat train.log | grep "loss" | wc -l

# Extract just the loss values from training output

grep "loss:" train.log | awk '{print $NF}' > losses.txt

# Watch a log file update in real time, filtering for errors

tail -f train.log | grep --line-buffered "ERROR"

# Sort experiments by final accuracy

grep "final_accuracy" results/*.log | sort -t= -k2 -n -r

# Redirect stdout and stderr to separate files

python train.py > output.log 2> errors.log

# Redirect both to the same file

python train.py > train_full.log 2>&1

In [ ]:
```

The three redirects you need:

| Symbol | What it does |

|--------|-------------|

| `>` | Write stdout to file (overwrite) |

| `>>` | Append stdout to file |

| `2>` | Write stderr to file |

| `2>&1` | Send stderr to same place as stdout |

| `\|` | Send stdout of one command as stdin to the next |

### Step 3: Background processes

Training runs take hours. You don't want to keep your terminal open the whole time.

In [ ]:
```bash

# Run in background (output still goes to terminal)

python train.py &

# Run in background, immune to hangup (closing terminal won't kill it)

nohup python train.py > train.log 2>&1 &

# Check what's running in background

jobs

ps aux | grep train.py

# Bring a background job to foreground

fg %1

# Kill a background process

kill %1

# or find its PID and kill that

kill $(pgrep -f "train.py")

In [ ]:
```

The difference between `&`, `nohup`, and `screen`/`tmux`:

| Method | Survives terminal close? | Can reattach? |

|--------|-------------------------|---------------|

| `command &` | No | No |

| `nohup command &` | Yes | No (check log file) |

| `screen` / `tmux` | Yes | Yes |

For anything longer than a few minutes, use tmux.

### Step 4: tmux

tmux lets you create persistent terminal sessions with multiple panes. This is the single most useful tool for managing training runs.

In [ ]:
```bash

# Install

# macOS

brew install tmux

# Ubuntu

sudo apt install tmux

# Start a named session

tmux new -s training

# Split horizontally

# Ctrl+B then "

# Split vertically

# Ctrl+B then %

# Navigate between panes

# Ctrl+B then arrow keys

# Detach (session keeps running)

# Ctrl+B then d

# Reattach

tmux attach -t training

# List sessions

tmux ls

# Kill a session

tmux kill-session -t training

In [ ]:
```

A typical AI workflow session:

In [ ]:
```bash

tmux new -s train

# Pane 1: start training

python train.py --epochs 100 --lr 1e-4

# Ctrl+B, " to split, then run GPU monitor

watch -n1 nvidia-smi

# Ctrl+B, % to split vertically, tail the logs

tail -f logs/experiment.log

# Now detach with Ctrl+B, d

# SSH out, go get coffee, come back

# tmux attach -t train

In [ ]:
```

### Step 5: Monitoring with htop and nvtop

In [ ]:
```bash

# System processes (better than top)

htop

# GPU processes (if you have NVIDIA GPU)

# Install: sudo apt install nvtop (Ubuntu) or brew install nvtop (macOS)

nvtop

# Quick GPU check without nvtop

nvidia-smi

# Watch GPU usage update every second

watch -n1 nvidia-smi

# See which processes are using the GPU

nvidia-smi --query-compute-apps=pid,name,used_memory --format=csv

In [ ]:
```

`htop` keybindings you'll use:

- `F6` or `>` to sort by column (sort by memory to find memory leaks)

- `F5` to toggle tree view (see child processes)

- `F9` to kill a process

- `/` to search for a process name

### Step 6: SSH for remote GPU boxes

When you rent a cloud GPU (Lambda, RunPod, Vast.ai), you connect via SSH.

In [ ]:
```bash

# Basic connection

ssh user@gpu-box-ip

# With a specific key

ssh -i ~/.ssh/my_gpu_key user@gpu-box-ip

# Copy files to remote

scp model.pt user@gpu-box-ip:~/models/

# Copy files from remote

scp user@gpu-box-ip:~/results/metrics.json ./

# Sync a whole directory (faster for many files)

rsync -avz ./data/ user@gpu-box-ip:~/data/

# Port forward (access remote Jupyter/TensorBoard locally)

ssh -L 8888:localhost:8888 user@gpu-box-ip

# Now open localhost:8888 in your browser

# SSH config for convenience

# Add to ~/.ssh/config:

# Host gpu

#     HostName 192.168.1.100

#     User ubuntu

#     IdentityFile ~/.ssh/gpu_key

#

# Then just:

# ssh gpu

In [ ]:
```

### Step 7: Useful aliases for AI work

Add these to your `~/.bashrc` or `~/.zshrc`:

In [ ]:
```bash

source phases/00-setup-and-tooling/10-terminal-and-shell/code/shell_aliases.sh

In [ ]:
```

Or copy the ones you want. The key aliases:

In [ ]:
```bash

# GPU status at a glance

alias gpu='nvidia-smi --query-gpu=index,name,utilization.gpu,memory.used,memory.total,temperature.gpu --format=csv,noheader'

# Kill all Python training processes

alias killtraining='pkill -f "python.*train"'

# Quick virtual environment activate

alias ae='source .venv/bin/activate'

# Watch training loss

alias watchloss='tail -f logs/*.log | grep --line-buffered "loss"'

In [ ]:
```

See `code/shell_aliases.sh` for the full set.

### Step 8: Common AI terminal patterns

These come up repeatedly in practice:

In [ ]:
```bash

# Run training, log everything, notify when done

python train.py 2>&1 | tee train.log; echo "DONE" | mail -s "Training complete" you@email.com

# Compare two experiment logs side by side

diff <(grep "accuracy" exp1.log) <(grep "accuracy" exp2.log)

# Find the largest model files (clean up disk space)

find . -name "*.pt" -o -name "*.safetensors" | xargs du -h | sort -rh | head -20

# Download a model from Hugging Face

wget https://huggingface.co/model/resolve/main/model.safetensors

# Untar a dataset

tar xzf dataset.tar.gz -C ./data/

# Count lines in all Python files (see how big your project is)

find . -name "*.py" | xargs wc -l | tail -1

# Check disk space (training data fills disks fast)

df -h

du -sh ./data/*

# Environment variable check before training

env | grep -i cuda

env | grep -i torch

In [ ]:
```

## Exercises

In [ ]:
1. Install tmux, create a session with three panes, and run `htop` in one, `watch -n1 date` in another, and a Python script in the third. Detach and reattach.
2. Add the aliases from `code/shell_aliases.sh` to your shell config and reload with `source ~/.zshrc` (or `~/.bashrc`).
3. Create a fake training log with `for i in $(seq 1 100); do echo "epoch $i loss: $(echo "scale=4; 1/$i" | bc)"; sleep 0.1; done > fake_train.log` and then use `grep`, `tail`, and `awk` to extract just the loss values.
4. Set up an SSH config entry for a server you have access to (or use `localhost` to practice the syntax).